In [ ]:
# ---------- Cell 3/3 ----------
# Run this last in Colab to start the bot.

import nest_asyncio
nest_asyncio.apply()

from telegram import Update
from telegram.ext import Application, CommandHandler, MessageHandler, filters
from datetime import datetime

# Use token from Cell 1
TELEGRAM_BOT_TOKEN = BotFather_API_Key

# Start command - shows format
async def start(update: Update, context):
    user = update.effective_user
    help_text = (
        f"Hi {user.mention_html()}! 👋\n"
        "Send your stock idea in the following format (use blank lines between sections):\n\n"
        "<pre>STOCK NAME\n\nENTRY_LOW-ENTRY_HIGH\n\nTARGET1\nTARGET2\nTARGET3\n\nSTOPLOSS\n\nSOURCE\n\nTYPE</pre>\n\n"
        "Example:\n"
        "<pre>state bank of india\n\n100-110\n\n120\n130\n140\n\n90\n\nAshish\n\nswing</pre>"
    )
    await update.message.reply_html(help_text)
    print(f"Bot started by {user.full_name} (ID: {user.id})")

# Message handler - parse and append to sheet
async def handle_message(update: Update, context):
    user_name = update.effective_user.full_name if update.effective_user else "Unknown"
    message_text = (update.message.text or "").strip()
    timestamp = now_ist_str()

    parsed, error = parse_stock_message_v2(message_text)
    if parsed:
        data_row = [timestamp, user_name] + parsed
        append_to_sheet(data_row)
        reply_text = (
            f"✅ Saved trade idea for *{parsed[0]}*.\n"
            "Data stored successfully in Google Sheet.\n\n"
            "📋 Format Reminder:\n"
            "```\nSTOCK NAME\n\nENTRY_LOW-ENTRY_HIGH\n\nTARGET1\nTARGET2\nTARGET3\n\nSTOPLOSS\n\nSOURCE\n\nTYPE\n```"
        )
        await update.message.reply_text(reply_text, parse_mode="Markdown")
        print(f"Processed structured message from {user_name}: {parsed}")
    else:
        await update.message.reply_text(
            f"{error}\n\nPlease use the exact format shown in /start.",
            parse_mode="Markdown",
        )
        print(f"Rejected invalid message from {user_name}: {message_text}")

# Build and run the application
def main():
    app = Application.builder().token(TELEGRAM_BOT_TOKEN).build()
    app.add_handler(CommandHandler("start", start))
    app.add_handler(MessageHandler(filters.TEXT & ~filters.COMMAND, handle_message))

    print("🤖 Bot is running... Send /start in Telegram to begin.")
    print("All entries will be logged to your Google Sheet:", GOOGLE_SHEET_NAME)
    app.run_polling(drop_pending_updates=True, close_loop=False, allowed_updates=Update.ALL_TYPES)

if __name__ == "__main__":
    main()
